In [ ]:
%pip install langchain langchain-openai langchain-text-splitters langchain-core pdfplumber python-dotenv openai numpy

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pdfplumber
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv
import os
from openai import OpenAI
import numpy as np

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

In [2]:
# Aplicar OCR
with pdfplumber.open("data/rag.pdf") as pdf:
    data_ocr = "\n".join([page.extract_text() for page in pdf.pages if page.extract_text()])
print (data_ocr)

REPORTE FINANCIERO ANUAL
La empresa ha experimentado un año de resultados financieros mixtos, reflejando tanto
oportunidades de crecimiento como desafíos en diversas áreas de operación. Los
ingresos han mostrado una tendencia de crecimiento moderado en comparación con el
año anterior, alcanzando un total de $250 millones, en comparación con los $232
millones registrados el año previo, lo que representa un incremento del 7.8%. El costo
de ventas ascendió a $162 millones, generando una utilidad bruta de $88 millones y un
margen bruto del 35.2%. Este crecimiento ha sido impulsado por una mayor demanda
en ciertos sectores del mercado, la expansión de la cartera de clientes y la
implementación de estrategias de optimización de ventas. Asimismo, la empresa ha
fortalecido sus canales de distribución y ha mejorado sus procesos comerciales, lo que
ha contribuido a mejorar el volumen de ventas y la participación en el mercado.
Sin embargo, la presión inflacionaria, con un aumento del 6.2% en los

In [3]:
# Configurar la división del texto
divisor_texto = RecursiveCharacterTextSplitter (
    chunk_size = 900,
    chunk_overlap = 200
)

In [5]:
# Se divide en chunks
chunks = divisor_texto.split_text(data_ocr)

In [6]:
client = OpenAI()

def embedding (texto):
    response = client.embeddings.create(
        input=texto,
        model = "text-embedding-ada-002"
    )

    return response.data[0].embedding

In [ ]:
# Se genera los embeddings
chunk_embedding = []
for chunk in chunks:
    chunk_embedding.append(embedding(chunk))

In [ ]:
# Comparación semántica
pregunta = "¿Cuánto fueron los ingresos totales de la empresa y cuánto crecieron respecto al año anterior?"
#pregunta = "¿Cuál fue la utilidad neta de la empresa al final del ejercicio?"
#pregunta = "¿Cuáles son las tres principales áreas donde se concentran los gastos de la empresa?"
#pregunta = "¿Cuál es la situación financiera de la empresa en términos de activos, pasivos y patrimonio?"
#pregunta = "¿Cuánto invirtió la empresa en maquinaria, tecnología y transformación digital?"

pregunta_embedding = embedding(pregunta)

similitudes = []
for chunk_emb in chunk_embedding:
    vec1, vec2 = np.array(pregunta_embedding), np.array(chunk_emb)
    similitud_cosenos = np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
    similitudes.append(similitud_cosenos)

In [ ]:
# Escoger los mejores chunks
cantidad = 4
top_indices = np.argsort(similitudes)[-cantidad:][::-1]

top_chunks = [chunks[i] for i in top_indices]
top_scores = [similitudes[i] for i in top_indices]

In [ ]:
# Se escogen los mejores chunks
cantidad = 5
top_indices = np.argsort(similitudes)[-cantidad:][::-1]

# Se aplica el umbral sobre los mejores chunks
umbral = 0.7
filtered_indices = [i for i in top_indices if similitudes[i] > umbral]

# Se obtiene los chunks y sus puntajes
top_chunks = [chunks[i] for i in filtered_indices]
top_scores = [similitudes[i] for i in top_indices]

In [ ]:
# Se instancia el LLM
llm = ChatOpenAI(
    model = "gpt-4o",
    temperature = 0.1,
    max_tokens = None,
    timeout = None,
    max_retries = 2,
    api_key = api_key
)

system_prompt = """
Eres un asistente experto en finanzas empresariales, especializado en el ánalisis de reportes financieros anuales.
Tu tarea es proporcionar respuestas precisas SOLO utilizando documentos referentes a reportes financieros que se te está compartiendo.
Los documentos utilizados para generar las respuestas estan delimitados por los caractéres ####.

Documentos:
####
{0}
####

IMPORTANTE:
- En caso de no saber la respuesta, no intentar responder con datos generados.
- En caso no se encuentre la respuesta en el documento, SOLO colocar: No hay información.
- La respuesta generada deberá ser detallada y en un lenguaje formal.
- La respuesta que generes no debe tener  más de 80 palabras.
"""

user_prompt = """
Responder a la pregunta utilizando información de los documentos compartidos.

Pregunta:
{0}
"""

In [ ]:
system_message = SystemMessage(content = system_prompt.format(top_chunks))
user_message = HumanMessage(content=user_prompt.format(pregunta))
response = llm.invoke([system_message, user_message])
print("Pregunta: ")
print(pregunta)
print("\n")
print("Respuesta IA: ")
print(response.content)